# Track 5: Group Relative Policy Optimization (GRPO) for Mathematical Reasoning

This notebook demonstrates fine-tuning **Qwen2.5-0.5B-Instruct** on mathematical reasoning problems from the `openai/gsm8k` dataset using **Group Relative Policy Optimization (GRPO)** over 250 steps. We configure reinforcement learning rewards based on format correctness (1.0 weight) and solution accuracy (3.0 weight) to teach the model strict XML reasoning and answer tags (`<reasoning>` and `<answer>`).

## 1. Setup Environment and Imports
We load libraries and establish system prompt instructions enforcing a strict XML reasoning and answer format.

In [1]:
import re

import os

import sys

import torch

import warnings

from datasets import load_dataset

from transformers import AutoTokenizer, AutoModelForCausalLM

from peft import LoraConfig

from trl import GRPOTrainer, GRPOConfig


warnings.filterwarnings("ignore")

os.environ["TOKENIZERS_PARALLELISM"] = "false"


compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
    else torch.float16
)

print(f"CUDA available: {torch.cuda.is_available()} | Compute Dtype: {compute_dtype}")

W0722 07:03:58.468000 1169340 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


W0722 07:03:58.483000 1169340 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


CUDA available: True | Compute Dtype: torch.bfloat16


## 2. Dataset Preparation (GSM8K)
We load the GSM8K dataset. We format each question with a system instruction that instructs the model to put its reasoning inside `<reasoning>` tags and the final numeric answer inside `<answer>` tags.

In [2]:
SYSTEM_PROMPT = (
    "A conversation between User and Assistant. The user asks a question, and the Assistant solves it.\n"
    "The assistant first thinks about the reasoning process in the mind and then provides the user with the answer.\n"
    "The reasoning process and answer are enclosed within tags. The answer must be a single integer.\n"
    "Example:\n"
    "<reasoning>\n"
    "We know that 2 + 2 = 4.\n"
    "</reasoning>\n"
    "<answer>4</answer>"
)


def extract_hash_answer(text: str) -> str | None:

    if "####" not in text:
        return None

    return text.split("####")[1].strip()


dataset = load_dataset("openai/gsm8k", "main", split="train")


# Select small subset of 300 prompts for fast notebook execution


dataset = dataset.shuffle(seed=42).select(range(800))


def format_gsm8k(example):

    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["question"]},
        ],
        "answer": extract_hash_answer(example["answer"]),
    }


gsm8k_train = dataset.map(format_gsm8k)


print("Sample Prompt Structure Preview:\n", gsm8k_train[0]["prompt"])

Sample Prompt Structure Preview:
 [{'role': 'system', 'content': 'A conversation between User and Assistant. The user asks a question, and the Assistant solves it.\nThe assistant first thinks about the reasoning process in the mind and then provides the user with the answer.\nThe reasoning process and answer are enclosed within tags. The answer must be a single integer.\nExample:\n<reasoning>\nWe know that 2 + 2 = 4.\n</reasoning>\n<answer>4</answer>'}, {'role': 'user', 'content': 'Mimi picked up 2 dozen seashells on the beach.  Kyle found twice as many shells as Mimi and put them in his pocket. Leigh grabbed one-third of the shells that Kyle found.  How many seashells did Leigh have?'}]


## 3. Define RL Rewards
We define two reward functions:
1. **Format Reward**: Returns `1.0` if the output strictly matches the `<reasoning>...</reasoning>\n<answer>...</answer>` tags.
2. **Correctness Reward**: Returns `2.0` if the extracted answer matches the ground truth, and `0.0` otherwise.

In [ ]:
def extract_last_number(text, start_tag="<answer>", end_tag="</answer>"):
    pattern = re.escape(start_tag) + r"(.*?)" + re.escape(end_tag)
    matches = re.findall(pattern, text, re.DOTALL)
    text_to_search = matches[-1] if matches else text
    numbers = re.findall(r"-?\d+(?:,\d{3})*(?:\.\d+)?", text_to_search)
    if numbers:
        return numbers[-1].replace(",", "").strip()
    return ""


def format_reward(completions, **kwargs):
    pattern = r"^<reasoning>[\s\S]*?<\/reasoning>\s*<answer>[\s\S]*?<\/answer>$"
    responses = [completion[0]["content"] for completion in completions]
    rewards = [1.0 if re.match(pattern, response) else 0.0 for response in responses]
    return rewards


def correctness_reward(completions, answer, **kwargs):
    responses = [completion[0]["content"] for completion in completions]
    extracted = [extract_last_number(response) for response in responses]
    rewards = [3.0 if ext == ans else 0.0 for ext, ans in zip(extracted, answer)]
    return rewards

## 3.5. Evaluate Pre-Trained Model Before GRPO Alignment
To demonstrate GRPO's capability to teach both format compliance and mathematical accuracy, we evaluate the un-finetuned `Qwen2.5-0.5B-Instruct` model on 5 test questions from GSM8K before starting RL training.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

base_model_eval = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct", dtype=compute_dtype, device_map="auto"
)

base_model_eval.eval()

base_model_eval.generation_config.max_length = None


eval_dataset_100 = load_dataset("openai/gsm8k", "main", split="test").select(range(100))

pre_grpo_responses_100 = []

pre_grpo_correct = 0

pre_grpo_format = 0


pattern = r"^<reasoning>[\s\S]*?<\/reasoning>\s*<answer>[\s\S]*?<\/answer>$"


print("=== Running Pre-GRPO Benchmark on 100 GSM8K Test Problems ===")

for i, example in enumerate(eval_dataset_100):
    question = example["question"]

    ground_truth = extract_hash_answer(example["answer"])

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    )

    input_ids = inputs if isinstance(inputs, torch.Tensor) else inputs["input_ids"]

    input_ids = input_ids.to("cuda")

    with torch.no_grad():
        outputs = base_model_eval.generate(
            input_ids=input_ids, max_new_tokens=384, pad_token_id=tokenizer.eos_token_id
        )

    resp = tokenizer.decode(
        outputs[0][input_ids.shape[1] :], skip_special_tokens=True
    ).strip()

    pre_grpo_responses_100.append(resp)

    ext_ans = extract_last_number(resp)

    if ext_ans == ground_truth:
        pre_grpo_correct += 1

    if re.match(pattern, resp):
        pre_grpo_format += 1

    if (i + 1) % 20 == 0:
        print(
            f"Evaluated {i + 1}/100 problems | Current Accuracy: {pre_grpo_correct / (i + 1) * 100:.1f}%"
        )


pre_acc = (pre_grpo_correct / 100) * 100

pre_fmt = (pre_grpo_format / 100) * 100

print(f"\n>>> PRE-GRPO BENCHMARK (100 Problems) <<<")

print(f"Format Compliance: {pre_fmt:.1f}% ({pre_grpo_format}/100)")

print(f"Math Accuracy:     {pre_acc:.1f}% ({pre_grpo_correct}/100)\n")


# Free GPU VRAM

del base_model_eval

torch.cuda.empty_cache()

## 4. Run GRPOTrainer using vLLM
We load the tokenizer and define the LoRA parameters. We use `vLLM` inside the trainer with a configured device and VRAM footprint to scale policy rollouts rapidly.

In [5]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


peft_config = LoraConfig(
    lora_alpha=64,
    lora_dropout=0.0,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)


training_args = GRPOConfig(
    use_vllm=False,
    learning_rate=1e-5,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_steps=25,
    beta=0.005,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    bf16=(compute_dtype == torch.bfloat16),
    fp16=(compute_dtype == torch.float16),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    gradient_accumulation_steps=4,
    per_device_train_batch_size=2,
    num_generations=4,
    temperature=0.5,
    max_completion_length=384,
    max_steps=250,
    logging_steps=10,
    save_steps=50,
    max_grad_norm=0.1,
    report_to="none",
    output_dir="qwen2.5-0.5b-grpo-output",
)


model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=compute_dtype, device_map="auto"
)

if not hasattr(model, "warnings_issued"):
    model.warnings_issued = {}


trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[correctness_reward, format_reward],
    args=training_args,
    train_dataset=gsm8k_train,
    peft_config=peft_config,
)


trainer.train()


merged_model = trainer.model.merge_and_unload()

tokenizer.save_pretrained("qwen2.5-0.5b-grpo-adapter")

merged_model.save_pretrained("qwen2.5-0.5b-grpo-adapter")

print("GRPO training completed and model saved!")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,0.009581
20,0.011159
30,0.039516
40,-0.068905
50,0.105207
60,0.013550
70,0.024280
80,0.018187
90,-0.006942
100,0.046491


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

GRPO training completed and model saved!


## 5. Inference Verification
We query our trained model to verify that it generates reasoning and mathematical solutions structured under correct tags.

In [ ]:
merged_model.eval()

merged_model.generation_config.max_length = None


post_grpo_responses_100 = []

post_grpo_correct = 0

post_grpo_format = 0


print("=== Running Post-GRPO Benchmark on 100 GSM8K Test Problems ===")

for i, example in enumerate(eval_dataset_100):
    question = example["question"]

    ground_truth = extract_hash_answer(example["answer"])

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    )

    input_ids = inputs if isinstance(inputs, torch.Tensor) else inputs["input_ids"]

    input_ids = input_ids.to("cuda")

    with torch.no_grad():
        outputs = merged_model.generate(
            input_ids=input_ids, max_new_tokens=384, pad_token_id=tokenizer.eos_token_id
        )

    post_resp = tokenizer.decode(
        outputs[0][input_ids.shape[1] :], skip_special_tokens=True
    ).strip()

    post_grpo_responses_100.append(post_resp)

    ext_ans = extract_last_number(post_resp)

    if ext_ans == ground_truth:
        post_grpo_correct += 1

    if re.match(pattern, post_resp):
        post_grpo_format += 1

    if (i + 1) % 20 == 0:
        print(
            f"Evaluated {i + 1}/100 problems | Current Accuracy: {post_grpo_correct / (i + 1) * 100:.1f}%"
        )


post_acc = (post_grpo_correct / 100) * 100

post_fmt = (post_grpo_format / 100) * 100


acc_delta = post_acc - pre_acc

fmt_delta = post_fmt - pre_fmt


print("\n" + "=" * 60)

print(">>> FINAL GSM8K 100-PROBLEM BENCHMARK RESULTS <<<")

print(f"Pre-GRPO Format Compliance:  {pre_fmt:.1f}%")

print(f"Post-GRPO Format Compliance: {post_fmt:.1f}%")

print(f"Format Compliance Delta:    {'+' if fmt_delta >= 0 else ''}{fmt_delta:.1f}%\n")

print(f"Pre-GRPO Math Accuracy:      {pre_acc:.1f}%")

print(f"Post-GRPO Math Accuracy:     {post_acc:.1f}%")

print(f"Math Accuracy Delta:        {'+' if acc_delta >= 0 else ''}{acc_delta:.1f}%")

print("=" * 60 + "\n")


print("=== Detailed 5-Example Side-by-Side Comparison ===")

for i in range(5):
    example = eval_dataset_100[i]

    question = example["question"]

    ground_truth = extract_hash_answer(example["answer"])

    print(f"=== Test Example {i + 1} ===")

    print(f"Question: {question}")

    print(f"Ground Truth Answer: {ground_truth}")

    print(
        f"\033[91mBefore GRPO (Pre-trained 0.5B Model):\033[0m\n{pre_grpo_responses_100[i]}\n"
    )

    print(
        f"\033[92mAfter GRPO (RL Aligned 0.5B Model):\033[0m\n{post_grpo_responses_100[i]}\n"
    )

    print("-" * 80 + "\n")